# NiyamTrace-X Q1 Experiment 08 — Risk-Calibrated Selective Authorization

**Purpose.** Explore a statistically calibrated layer that predicts when the *raw* semantic-to-effect policy is at elevated risk and routes those cases to clarification/secondary verification rather than automatically allowing them.

This notebook is intentionally conservative:

- It uses only **deployment-observable features** from the model output, authenticated contract and deterministic Anchor analysis.
- It explicitly forbids ground-truth-only fields such as `expected_verdict`, `relation`, `risk`, `semantic_correct`, or slot-match labels from the feature matrix.
- Semantic groups, not individual languages, are split into train/calibration/test sets to prevent cross-language leakage.
- A logistic risk model is trained on the training groups.
- The **calibration** groups choose the largest auto-ALLOW threshold whose one-sided Clopper–Pearson upper confidence bound satisfies a target harmful-ALLOW risk.
- The untouched test groups report achieved risk, coverage, calibration and language/relation slices.

This is a research extension, not a claim of distribution-free safety under arbitrary deployment shift. The notebook also compares the learned selective gate with the deterministic frozen Anchor Lock.

**Important ablation.** The notebook reports an Anchor-aware risk model and two reduced models (`no_anchor` and `semantic_only`) so a perfect score caused by simply relearning the deterministic Anchor Lock is not mistaken for a new uncertainty contribution.


In [ ]:
import importlib.util, subprocess, sys
need=[p for p in ['pandas','numpy','scipy','scikit-learn','matplotlib'] if importlib.util.find_spec(p) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q']+need)


In [ ]:
from pathlib import Path
import os, json, math, hashlib, zipfile, shutil, random, statistics, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260911
random.seed(SEED)
np.random.seed(SEED)

BASE = Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS = BASE / 'niyamtrace_q1_wave2_results'
RESULTS.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR = BASE / 'ntx_frozen_evidence'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'holdout2000.jsonl': '4dad5dad9ea4a3258f6139407664ab2584678b440622871fa1b5d5da24f95d3a',
    'qwen_results.jsonl': '7d1b3bf2b0b34d0101c1a885847d6d511a54014c16f43e4cea778b6078123eb7',
    'gptoss_results.jsonl': '991812849fb9f5a0651b6277b7bc71d6c2c1ca8796b166eb182ec8fac221e81d',
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def find_or_upload_evidence():
    candidates = [
        BASE/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
        Path.cwd()/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
    ]
    for p in candidates:
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip')
        uploaded = files.upload()
        for name, data in uploaded.items():
            p=BASE/name
            p.write_bytes(data)
            if name.endswith('.zip'): return p
    except Exception as e:
        raise FileNotFoundError('Place NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip in /content or current directory.') from e
    raise FileNotFoundError('Evidence ZIP not found.')

ZIP = find_or_upload_evidence()
with zipfile.ZipFile(ZIP) as z:
    z.extractall(EVIDENCE_DIR)

# Accept either flat package or a nested folder.
def locate(name):
    hits=list(EVIDENCE_DIR.rglob(name))
    if len(hits)!=1:
        raise RuntimeError(f'Expected exactly one {name}, found {len(hits)}: {hits[:5]}')
    return hits[0]

paths={k:locate(k) for k in EXPECTED}
for name, exp in EXPECTED.items():
    got=sha256(paths[name])
    print(name, got, 'OK' if got==exp else 'HASH MISMATCH')
    assert got==exp, (name, got, exp)

holdout=pd.read_json(paths['holdout2000.jsonl'], lines=True)
qwen=pd.read_json(paths['qwen_results.jsonl'], lines=True)
gpt=pd.read_json(paths['gptoss_results.jsonl'], lines=True)
assert len(holdout)==len(qwen)==len(gpt)==2000
assert holdout.variant_group_id.nunique()==500
assert set(qwen.case_id)==set(holdout.case_id)==set(gpt.case_id)
print('Evidence verified:', len(holdout), 'cases /', holdout.variant_group_id.nunique(), 'semantic groups')


In [ ]:
from scipy.stats import beta
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

FORBIDDEN={'expected_verdict','relation','risk','semantic_correct','semantic_exact_match','critical_match','critical_slot_accuracy','slot_matches','decision_correct','correct','unsafe_allow','false_block'}

def parse_uncertainty_count(raw):
    try:
        obj=json.loads(raw); return sum(len(v or []) for c in obj.get('candidates',[]) for v in (c.get('uncertainty') or {}).values())
    except Exception:return 0

def mismatch_count(top,auth):
    fields=['action','vendor_id','month','year','amount','currency','scope','negated']
    top=top if isinstance(top,dict) else {}; auth=auth if isinstance(auth,dict) else {}
    return sum(top.get(f)!=auth.get(f) for f in fields)

def features(df):
    X=pd.DataFrame(index=df.index)
    X['anchor_n']=df.anchor_violations.map(lambda x:len(x) if isinstance(x,list) else 0)
    X['num_candidates']=df.num_candidates.fillna(0).astype(float)
    X['needs_clarification']=df.needs_clarification.astype(int)
    X['model_disagreement_n']=df.model_disagreements.map(lambda x:len(x) if isinstance(x,list) else 0)
    X['effective_disagreement_n']=df.effective_disagreements.map(lambda x:len(x) if isinstance(x,list) else 0)
    X['candidate_authorized_mismatch_n']=[mismatch_count(t,a) for t,a in zip(df.top_candidate,df.authorized_contract)]
    X['uncertainty_value_n']=df.raw_model_content.map(parse_uncertainty_count)
    X['schema_valid']=df.schema_valid.astype(int)
    X['raw_is_allow']=(df.raw_policy_verdict=='ALLOW').astype(int)
    X['raw_is_clarify']=(df.raw_policy_verdict=='CLARIFY').astype(int)
    X['raw_is_block']=(df.raw_policy_verdict=='BLOCK').astype(int)
    assert not (set(X.columns)&FORBIDDEN)
    return X

def harmful_raw_allow(df): return ((df.raw_policy_verdict=='ALLOW') & (df.expected_verdict!='ALLOW')).astype(int)

X=features(qwen); y=harmful_raw_allow(qwen)
print('Qwen raw unsafe ALLOW events:',int(y.sum()),'of',len(y))
display(X.describe().T)


In [ ]:
# Group-disjoint train/calibration/test split: 200/150/150 semantic groups.
rng=np.random.default_rng(SEED); groups=np.array(sorted(qwen.variant_group_id.unique())); rng.shuffle(groups)
train_g=set(groups[:200]); cal_g=set(groups[200:350]); test_g=set(groups[350:])
train=qwen.variant_group_id.isin(train_g); cal=qwen.variant_group_id.isin(cal_g); test=qwen.variant_group_id.isin(test_g)
assert not (train & cal).any() and not (train & test).any() and not (cal & test).any()
assert train.sum()==800 and cal.sum()==600 and test.sum()==600

model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced',random_state=SEED))
model.fit(X[train],y[train])
for name,mask in [('train',train),('calibration',cal),('test',test)]:
    p=model.predict_proba(X[mask])[:,1]
    auc=roc_auc_score(y[mask],p) if y[mask].nunique()>1 else np.nan
    print(name,'events=',int(y[mask].sum()),'AUC=',round(float(auc),4),'Brier=',round(float(brier_score_loss(y[mask],p)),5))

q=qwen.copy(); q['risk_score']=model.predict_proba(X)[:,1]


In [ ]:
# Feature-source ablation: quantify how much predictive signal remains without simply relearning Anchor Lock.
feature_sets={
    'anchor_aware': list(X.columns),
    'no_anchor': [c for c in X.columns if c!='anchor_n'],
    'semantic_only': [c for c in X.columns if c not in {'anchor_n','candidate_authorized_mismatch_n'}],
}
model_ablation=[]
for name,cols in feature_sets.items():
    m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced',random_state=SEED))
    m.fit(X.loc[train,cols],y[train])
    for split,mask in [('calibration',cal),('test',test)]:
        pp=m.predict_proba(X.loc[mask,cols])[:,1]
        model_ablation.append({
            'feature_set':name,'split':split,'n_features':len(cols),
            'auc':float(roc_auc_score(y[mask],pp)) if y[mask].nunique()>1 else np.nan,
            'brier':float(brier_score_loss(y[mask],pp)),
            'harmful_events':int(y[mask].sum()),
        })
model_ablation=pd.DataFrame(model_ablation)
display(model_ablation)
model_ablation.to_csv(RESULTS/'exp08_feature_source_ablation.csv',index=False)


In [ ]:
# Risk-controlling threshold on calibration groups.
def cp_upper(events,n,alpha=.05):
    if n==0:return 1.0
    if events==n:return 1.0
    return float(beta.ppf(1-alpha,events+1,n-events))

TARGET_RISK=.05  # 5% one-sided upper bound on harmful auto-ALLOW among selected raw-ALLOW cases.
caldf=q[cal & (q.raw_policy_verdict=='ALLOW')].copy()
thresholds=np.unique(np.r_[0,caldf.risk_score.values,1])
curve=[]
for t in thresholds:
    sel=caldf.risk_score<=t; n=int(sel.sum()); e=int(((caldf.loc[sel].expected_verdict!='ALLOW')).sum())
    curve.append({'threshold':float(t),'n_auto_allow':n,'harmful':e,'empirical_risk':e/n if n else np.nan,'cp95_upper':cp_upper(e,n),'coverage_within_raw_allow':n/len(caldf) if len(caldf) else 0})
curve=pd.DataFrame(curve)
valid=curve[(curve.n_auto_allow>0)&(curve.cp95_upper<=TARGET_RISK)]
if len(valid): chosen=valid.sort_values(['coverage_within_raw_allow','threshold']).iloc[-1]
else: chosen=curve.sort_values(['cp95_upper','coverage_within_raw_allow'],ascending=[True,False]).iloc[0]
T=float(chosen.threshold)
print('Chosen threshold:',T); display(chosen.to_frame().T)
curve.to_csv(RESULTS/'exp08_calibration_risk_coverage_curve.csv',index=False)


In [ ]:
# Untouched test evaluation. High-risk raw ALLOWs are routed to CLARIFY.
def selective_verdicts(df,t):
    out=df.raw_policy_verdict.copy()
    mask=(df.raw_policy_verdict=='ALLOW') & (df.risk_score>t)
    out.loc[mask]='CLARIFY'
    return out

def report(df,pred,name):
    exp=df.expected_verdict
    raw_allow=(df.raw_policy_verdict=='ALLOW')
    selected=(pred=='ALLOW')
    return {
        'policy':name,'n':len(df),'accuracy':float((pred==exp).mean()),
        'unsafe_allow':int(((pred=='ALLOW')&(exp!='ALLOW')).sum()),
        'false_block':int(((pred=='BLOCK')&(exp=='ALLOW')).sum()),
        'unnecessary_clarify':int(((pred=='CLARIFY')&(exp=='ALLOW')).sum()),
        'auto_allow_count':int(selected.sum()),'auto_allow_rate':float(selected.mean()),
        'raw_allow_retention':float((selected&raw_allow).sum()/raw_allow.sum()) if raw_allow.sum() else np.nan,
    }

t=q[test].copy(); sel=selective_verdicts(t,T)
rows=[report(t,t.raw_policy_verdict,'raw_policy'),report(t,sel,'risk_selective'),report(t,t.actual_verdict,'frozen_full_runtime')]
res=pd.DataFrame(rows); display(res); res.to_csv(RESULTS/'exp08_test_policy_comparison.csv',index=False)

# One-sided upper bound for harmful auto-ALLOW among selective ALLOWs on test.
e=int(((sel=='ALLOW')&(t.expected_verdict!='ALLOW')).sum()); n=int((sel=='ALLOW').sum())
print('Test selective auto-ALLOW harmful events:',e,'/',n,'CP95 upper=',cp_upper(e,n))


In [ ]:
# Test slices are descriptive only; language/relation/risk were NOT training features.
slice_rows=[]
for col in ['language','relation','risk']:
    for key,gidx in t.groupby(col).groups.items():
        g=t.loc[gidx]; pred=sel.loc[gidx]
        rr=report(g,pred,'risk_selective'); slice_rows.append({'slice':col,'value':key,**rr})
slices=pd.DataFrame(slice_rows); slices.to_csv(RESULTS/'exp08_test_slices.csv',index=False); display(slices)


In [ ]:
# Repeated group-split stability audit (descriptive, 20 independent group partitions).
def one_split(seed):
    rng=np.random.default_rng(seed); gs=np.array(sorted(qwen.variant_group_id.unique())); rng.shuffle(gs)
    tr=set(gs[:200]); ca=set(gs[200:350]); te=set(gs[350:])
    mtr=qwen.variant_group_id.isin(tr); mca=qwen.variant_group_id.isin(ca); mte=qwen.variant_group_id.isin(te)
    mdl=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced',random_state=seed)); mdl.fit(X[mtr],y[mtr])
    qq=qwen.copy(); qq['risk_score']=mdl.predict_proba(X)[:,1]
    c=qq[mca & (qq.raw_policy_verdict=='ALLOW')]
    cs=[]
    for th in np.unique(np.r_[0,c.risk_score.values,1]):
        s=c.risk_score<=th; nn=int(s.sum()); ee=int((c.loc[s].expected_verdict!='ALLOW').sum()); cs.append((th,nn,ee,cp_upper(ee,nn),nn/len(c) if len(c) else 0))
    ok=[z for z in cs if z[1]>0 and z[3]<=TARGET_RISK]
    z=max(ok,key=lambda v:(v[4],v[0])) if ok else min(cs,key=lambda v:(v[3],-v[4]))
    td=qq[mte]; pred=selective_verdicts(td,float(z[0])); rr=report(td,pred,'risk_selective')
    rr.update({'seed':seed,'threshold':z[0],'cal_cp95_upper':z[3],'cal_coverage':z[4]}); return rr

stability=pd.DataFrame([one_split(SEED+i) for i in range(20)])
stability.to_csv(RESULTS/'exp08_repeated_group_split_stability.csv',index=False)
display(stability.describe().T)


In [ ]:
# Model-family transfer: apply Qwen-trained risk model to GPT-OSS observable features.
# This is descriptive transfer only; GPT has only two raw unsafe-ALLOW events overall.
Xg=features(gpt); gg=gpt.copy(); gg['risk_score']=model.predict_proba(Xg)[:,1]
gtest=gg[gg.variant_group_id.isin(test_g)].copy(); gpred=selective_verdicts(gtest,T)
gres=pd.DataFrame([report(gtest,gtest.raw_policy_verdict,'gpt_raw_policy'),report(gtest,gpred,'qwen_calibrated_transfer'),report(gtest,gtest.actual_verdict,'gpt_frozen_full')])
display(gres); gres.to_csv(RESULTS/'exp08_gpt_transfer.csv',index=False)


In [ ]:
# Risk–coverage and score-distribution figures.
fig,ax=plt.subplots(figsize=(7.5,4.5)); cc=curve[curve.n_auto_allow>0].sort_values('coverage_within_raw_allow'); ax.plot(cc.coverage_within_raw_allow,cc.cp95_upper,marker='.',linewidth=1); ax.axhline(TARGET_RISK,linestyle='--'); ax.axvline(float(chosen.coverage_within_raw_allow),linestyle=':'); ax.set_xlabel('Calibration auto-ALLOW coverage'); ax.set_ylabel('One-sided 95% risk upper bound'); ax.set_title('Risk-controlled selective authorization'); plt.tight_layout(); plt.savefig(RESULTS/'exp08_risk_coverage_curve.png',dpi=220,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(7.5,4.5)); safe=q[test & (y==0)]['risk_score']; harm=q[test & (y==1)]['risk_score']; ax.hist(safe,bins=20,alpha=.65,label='No harmful raw ALLOW'); ax.hist(harm,bins=20,alpha=.65,label='Harmful raw ALLOW'); ax.axvline(T,linestyle='--',label='calibrated threshold'); ax.set_xlabel('Predicted semantic-effect risk'); ax.set_ylabel('Cases'); ax.legend(); ax.set_title('Observable risk-score separation on held-out groups'); plt.tight_layout(); plt.savefig(RESULTS/'exp08_risk_score_distribution.png',dpi=220,bbox_inches='tight'); plt.show()


In [ ]:
# Feature coefficients, leakage audit, manuscript table and manifest.
clf=model.named_steps['logisticregression']; scaler=model.named_steps['standardscaler']; coef=pd.DataFrame({'feature':X.columns,'standardized_coefficient':clf.coef_[0]}).sort_values('standardized_coefficient',key=abs,ascending=False)
coef.to_csv(RESULTS/'exp08_feature_coefficients.csv',index=False); display(coef)

leak={'feature_columns':list(X.columns),'forbidden_overlap':sorted(set(X.columns)&FORBIDDEN),'group_split':{'train_groups':200,'calibration_groups':150,'test_groups':150},'target_risk':TARGET_RISK}
assert not leak['forbidden_overlap']
(RESULTS/'exp08_leakage_audit.json').write_text(json.dumps(leak,indent=2),encoding='utf-8')
(RESULTS/'exp08_policy_table.tex').write_text(res.to_latex(index=False,float_format=lambda x:f'{x:.4f}'),encoding='utf-8')
manifest={'experiment':'NTX_Q1_08_Risk_Calibrated_Selective_Authorization','seed':SEED,'input_sha256':EXPECTED,'feature_policy':'deployment-observable only','target_cp95_risk_upper':TARGET_RISK,'chosen_threshold':T,'warning':'Calibration guarantee is empirical for the held-out benchmark protocol and should not be generalized to arbitrary distribution shift.'}
(RESULTS/'exp08_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')


In [ ]:
# FINAL CELL — package this notebook's complete results and download the ZIP.
from pathlib import Path
import zipfile, hashlib

PREFIX = 'exp08_'
ZIP_OUT = BASE / 'NTX_Q1_08_RISK_CALIBRATED_AUTHORIZATION_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.iterdir()):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p, arcname=p.name)

sha = hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:', ZIP_OUT)
print('SHA-256:', sha)
print('Size MiB:', round(ZIP_OUT.stat().st_size/1024**2, 3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab; ZIP is available at:', ZIP_OUT)
